# COMETKiwi QE Scoring on DivEMT

This notebook computes reference-free quality estimation (QE) scores for machine-translated text using [COMETKiwi](https://huggingface.co/Unbabel/wmt22-cometkiwi-da), applied to the [DivEMT](https://huggingface.co/datasets/GroNLP/divemt) dataset.

**Pipeline:**
1. Load the DivEMT dataset (`main.tsv`)
2. Filter to machine-translated + post-edited rows (exclude human translation, `task_type == 'ht'`)
3. Build COMET input pairs (`src_text`, `mt_text`)
4. Score each pair with COMETKiwi
5. Populate dataset with QE score

In [6]:
import pandas as pd

pd.set_option("display.max_rows", None)      # show every row - dataset is small enough to inspect in full
pd.set_option("display.max_colwidth", None)  # never truncate cell text - need to read full src/mt strings

DATA_PATH = "divemt/main.tsv"
MODEL_NAME = "Unbabel/wmt22-cometkiwi-da"
OUTPUT_PATH = "divemt_cometkiwi_qe.csv"

## 1. Load the dataset

In [7]:
divemt_df = pd.read_csv(DATA_PATH, sep="\t")
divemt_df.dtypes

unit_id                     object
flores_id                    int64
item_id                     object
subject_id                  object
task_type                   object
translation_type            object
src_len_chr                  int64
mt_len_chr                 float64
tgt_len_chr                  int64
src_len_wrd                  int64
mt_len_wrd                 float64
tgt_len_wrd                  int64
edit_time                  float64
k_total                      int64
k_letter                     int64
k_digit                      int64
k_white                      int64
k_symbol                     int64
k_nav                        int64
k_erase                      int64
k_copy                       int64
k_cut                        int64
k_paste                      int64
k_do                         int64
n_pause_geq_300              int64
len_pause_geq_300            int64
n_pause_geq_1000             int64
len_pause_geq_1000           int64
event_time          

## 2. Filter to machine-translated rows

DivEMT contains three `task_type` values: `ht` (human translation from scratch), `pe1` (post-edited Google Translate output), and `pe2` (post-edited mBART50 output). COMETKiwi needs an existing machine translation to score, so we keep only the post-edited rows.

In [8]:
#The number of observations for each language based on task type
ht = divemt_df[divemt_df['task_type']=='ht']
mt = divemt_df[divemt_df['task_type']!='ht']
google = divemt_df[divemt_df['task_type']=='pe1']
mbart50 = divemt_df[divemt_df['task_type']=='pe2']

## 3. Build COMET input

COMETKiwi expects a list of `{"src": ..., "mt": ...}` dicts.

In [9]:
mt_df = mt[["src_text", "mt_text"]].rename(columns={"src_text": "src", "mt_text": "mt"})
data_mt = mt_df.to_dict(orient="records")
len(data_mt)

5160

## 4. Load the COMETKiwi model

In [10]:
from comet import download_model, load_from_checkpoint

model_path = download_model(MODEL_NAME)
comet_kiwi = load_from_checkpoint(model_path)

/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 16777.22it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.2 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../.cache/huggingface/hub/models--Unbabel--wmt22-cometkiwi-da/snapshots/1ad785194e391eebc6c53e2d0776cada8f83179a/checkpoints/model.ckpt`
Encoder model frozen.
/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/.venv/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


## 5. Score the translations


In [11]:
mtqe_score_cometkiwi = comet_kiwi.predict(data_mt, length_batching=False)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Predicting DataLoader 0: 100%|██████████| 323/323 [2:20:03<00:00, 26.02s/it]  


## 6. Attach QE score to dataset

In [12]:
cometkiwi_qe = mt.assign(cometkiwi_qe=mtqe_score_cometkiwi.scores)
cometkiwi_qe.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

In [19]:
test_df = pd.read_csv(OUTPUT_PATH)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5160 entries, 0 to 5159
Data columns (total 65 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   unit_id                  5160 non-null   object 
 1   flores_id                5160 non-null   int64  
 2   item_id                  5160 non-null   object 
 3   subject_id               5160 non-null   object 
 4   task_type                5160 non-null   object 
 5   translation_type         5160 non-null   object 
 6   src_len_chr              5160 non-null   int64  
 7   mt_len_chr               5160 non-null   float64
 8   tgt_len_chr              5160 non-null   int64  
 9   src_len_wrd              5160 non-null   int64  
 10  mt_len_wrd               5160 non-null   float64
 11  tgt_len_wrd              5160 non-null   int64  
 12  edit_time                5160 non-null   float64
 13  k_total                  5160 non-null   int64  
 14  k_letter                